# OCT anomaly detection: preprocessing (stage 04)

This notebook runs the same preprocessing as `04_preprocess.py` in the project repository, one step at a time, so every step can be seen.

**Pipeline per B-scan**
1. Convert to grayscale and black out the burned-in "200 µm" scale bar (bottom-left corner, rows 436–486).
2. Find the retina: Gaussian blur → Otsu threshold → morphological opening; per column, take the first and last retina pixel; the band runs from the 2nd percentile of the tops to the 98th percentile of the bottoms.
3. Crop a fixed window of **352 rows × full width (768)** centred on the retina (centre crop if detection fails).
4. Resize to **224 × 224** and store as uint8.

Dataset: *Labeled Retinal OCT Dataset for Classification of Normal, Drusen, and CNV Cases* (Noor Eye Hospital, Mendeley Data v2): 16,803 B-scans, 441 patients.

## 1. Setup

Run in **Google Colab** (Runtime → Run all). Everything used here is preinstalled in Colab.

In [ ]:
import os, re, io, zipfile, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage
from skimage.filters import threshold_otsu
from tqdm.auto import tqdm

plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 9})
SEED = 42
rng = np.random.default_rng(SEED)
print("numpy", np.__version__, "| pandas", pd.__version__)

## 2. Get the data

1. Upload the Mendeley download (`Labeled Retinal Optical Coherence Tomography Dataset for Classification of Normal, Drusen, and CNV Cases.zip`, 4.2 GB) to your Google Drive.
2. Set `ZIP_PATH` below to where it is in your Drive.

The cell copies the zip into Colab, extracts it, then extracts the image zip inside it (about 3–5 minutes). If the data is already extracted (or you run this locally), set `DATA_DIR` instead.

In [ ]:
ZIP_PATH = "/content/drive/MyDrive/Labeled Retinal Optical Coherence Tomography Dataset for Classification of Normal, Drusen, and CNV Cases.zip"
DATA_DIR = os.environ.get("OCT_DATA_DIR", "/content/data")      # folder that will hold NEH_UT_2021RetinalOCTDataset/

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False
if IN_COLAB and not Path(DATA_DIR, "NEH_UT_2021RetinalOCTDataset").exists():
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    with zipfile.ZipFile(ZIP_PATH) as outer:
        inner_name = next(n for n in outer.namelist() if n.lower().endswith(".zip"))
        csv_name = next(n for n in outer.namelist() if n.endswith("data_information.csv"))
        Path(DATA_DIR, "data_information.csv").write_bytes(outer.read(csv_name))
        print("extracting", inner_name.split("/")[-1], "...")
        outer.extract(inner_name, "/content/tmp")
    with zipfile.ZipFile(Path("/content/tmp", inner_name)) as inner:
        inner.extractall(DATA_DIR)
    print(f"done in {time.time() - t0:.0f}s")

ROOT = Path(DATA_DIR)
print("data folder:", ROOT.resolve(), "| contents:", sorted(p.name for p in ROOT.iterdir()))

## 3. Index the dataset

Filename rule: `CLASS/<case>/[OD|OS/]<bscan>_<Label>.<ext>`. Case numbers restart at 1 in every class folder, so the patient id is `CLASS-case` (e.g. `CNV-047`). `data_information.csv` gives the eye for every file.

In [ ]:
CLASSES = ["NORMAL", "DRUSEN", "CNV"]
EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
PATTERN = re.compile(r"(?P<cls>NORMAL|DRUSEN|CNV)/(?P<case>\d+)/(?:(?P<eye>OD|OS)/)?(?P<idx>\d+)_(?P<label>[A-Za-z]+)\.\w+$",
                     re.IGNORECASE)

rows = []
for p in ROOT.rglob("*"):
    if p.suffix.lower() not in EXTS:
        continue
    rel = p.as_posix()
    m = PATTERN.search(rel)
    if m is None:
        continue
    cls, case = m["cls"].upper(), int(m["case"])
    rows.append(dict(path=str(p), image_path=rel[m.start():], patient_id=f"{cls}-{case:03d}", class_folder=cls,
                     bscan_index=int(m["idx"]), bscan_label=m["label"].lower(), ext=p.suffix.lower()))
df = pd.DataFrame(rows).sort_values(["class_folder", "patient_id", "image_path"]).reset_index(drop=True)

info_csv = ROOT / "data_information.csv"
if info_csv.exists():
    info = pd.read_csv(info_csv).drop_duplicates("Directory").set_index("Directory")
    df["eye"] = df.image_path.map(info["Eye"])

print(f"{len(df):,} images from {df.patient_id.nunique()} patients   (full dataset: 16,803 images, 441 patients)")
display(df.groupby("class_folder").agg(patients=("patient_id", "nunique"), images=("path", "size")).reindex(CLASSES))
print("Specialist B-scan label vs. class folder (full dataset: 29 drusen slices in NORMAL, 2,910 normal slices in DRUSEN/CNV):")
display(pd.crosstab(df.class_folder, df.bscan_label).reindex(CLASSES))

## 4. Look at the raw images

All images are 768 × 496, grayscale stored as RGB. Every scan carries a burned-in "200 µm" scale bar in the bottom-left corner; it is the only coloured pixel content and is removed in step 1.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.4))
for ax, cls in zip(axes, CLASSES):
    r = df[df.class_folder == cls].sample(1, random_state=SEED).iloc[0]
    im = Image.open(r.path)
    ax.imshow(im, cmap="gray")
    ax.set_title(f"{r.image_path}\n{im.size[0]}x{im.size[1]}, mode {im.mode}")
    ax.axis("off")
axes[3].imshow(np.asarray(Image.open(r.path).convert("RGB"))[440:496, 0:120])
axes[3].set_title("zoom: scale bar (bottom-left)")
axes[3].axis("off")
plt.tight_layout(); plt.show()

## 5. The preprocessing, step by step

The same functions as `04_preprocess.py`, with one correction: the scale-bar mask covers rows 430–491. Measured over many scans, the bar spans rows 436–486 (columns 10–55); the mask in `04_preprocess.py` (rows 455–491) left the top of its vertical stroke in place.

In [ ]:
SCALE_BAR = (slice(430, 492), slice(0, 86))   # rows, cols of the burned-in scale bar (bar spans rows 436-486)
ROI_HEIGHT = 352                               # crop height (rows); full 768 width
SIZE = 224                                     # output size
MIN_COVERAGE, MIN_SPAN = 0.3, 60               # detection sanity limits


def load_gray(path):
    g = np.asarray(Image.open(path).convert("L")).copy()
    g[SCALE_BAR] = 0                           # step 1: remove the scale bar
    return g


def detect_retina(gray):
    """Step 2: (top, bottom, coverage) of the retinal band, or None if detection fails."""
    s = ndimage.gaussian_filter(gray.astype(np.float32), 4)
    mask = ndimage.binary_opening(s > threshold_otsu(s), iterations=2)
    cols = mask.any(axis=0)
    if cols.mean() < MIN_COVERAGE:
        return None
    tops = mask.argmax(axis=0)[cols]
    bottoms = (mask.shape[0] - 1 - mask[::-1].argmax(axis=0))[cols]
    top, bottom = np.percentile(tops, 2), np.percentile(bottoms, 98)
    if bottom - top < MIN_SPAN:
        return None
    return float(top), float(bottom), float(cols.mean())


def preprocess(gray):
    """Steps 3-4: crop ROI_HEIGHT rows centred on the retina, resize to SIZE x SIZE."""
    h = gray.shape[0]
    det = detect_retina(gray)
    if det is None:
        crop_top, info = (h - ROI_HEIGHT) // 2, dict(retina_top=np.nan, retina_bottom=np.nan, fallback=True)
    else:
        top, bottom, _ = det
        crop_top = int(round((top + bottom) / 2 - ROI_HEIGHT / 2))
        crop_top = min(max(crop_top, 0), h - ROI_HEIGHT)
        info = dict(retina_top=top, retina_bottom=bottom, fallback=False)
    out = np.asarray(Image.fromarray(gray[crop_top:crop_top + ROI_HEIGHT]).resize((SIZE, SIZE), Image.BILINEAR))
    info.update(crop_top=crop_top, crop_bottom=crop_top + ROI_HEIGHT)
    return out, info

In [ ]:
# walk one CNV scan through every step
r = df[df.class_folder == "CNV"].sample(1, random_state=7).iloc[0]
raw = np.asarray(Image.open(r.path).convert("L"))
gray = load_gray(r.path)
blur = ndimage.gaussian_filter(gray.astype(np.float32), 4)
mask = ndimage.binary_opening(blur > threshold_otsu(blur), iterations=2)
top, bottom, cov = detect_retina(gray)
out, info = preprocess(gray)

fig, ax = plt.subplots(2, 3, figsize=(15, 7))
ax[0, 0].imshow(raw, cmap="gray"); ax[0, 0].set_title(f"0. raw  {r.image_path}")
ax[0, 1].imshow(gray, cmap="gray"); ax[0, 1].set_title("1. grayscale, scale bar removed")
ax[0, 2].imshow(blur, cmap="gray"); ax[0, 2].set_title("2a. Gaussian blur (sigma 4)")
ax[1, 0].imshow(mask, cmap="gray"); ax[1, 0].set_title(f"2b. Otsu + opening: retina mask ({cov:.0%} of columns)")
ax[1, 1].imshow(gray, cmap="gray")
for y, c, ls in ((top, "#38BDF8", "--"), (bottom, "#38BDF8", "--"), (info["crop_top"], "#F59E0B", "-"), (info["crop_bottom"], "#F59E0B", "-")):
    ax[1, 1].axhline(y, color=c, linestyle=ls, linewidth=1.5)
ax[1, 1].set_title("3. retina band (blue) and 352-row crop window (orange)")
ax[1, 2].imshow(out, cmap="gray"); ax[1, 2].set_title(f"4. output {out.shape[1]}x{out.shape[0]}")
for a in ax.flat:
    a.axis("off")
plt.tight_layout(); plt.show()

## 6. Before / after on several scans

Orange = crop window, blue dashed = detected retina. The crop follows the retina even when the scan is tilted or the retina sits high or low in the frame.

In [ ]:
pick = pd.concat([df[df.class_folder == c].sample(min(4, (df.class_folder == c).sum()), random_state=SEED) for c in CLASSES])
fig, axes = plt.subplots(len(pick) // 4 * 2, 4, figsize=(16, 4.2 * len(pick) // 4))
for k, (_, r) in enumerate(pick.iterrows()):
    g = load_gray(r.path)
    o, inf = preprocess(g)
    a, b = axes[(k // 4) * 2, k % 4], axes[(k // 4) * 2 + 1, k % 4]
    a.imshow(g, cmap="gray", aspect="auto")
    a.axhline(inf["crop_top"], color="#F59E0B"); a.axhline(inf["crop_bottom"], color="#F59E0B")
    if not inf["fallback"]:
        a.axhline(inf["retina_top"], color="#38BDF8", ls="--"); a.axhline(inf["retina_bottom"], color="#38BDF8", ls="--")
    a.set_title(f"{r.image_path} ({r.bscan_label})")
    b.imshow(o, cmap="gray")
    for x in (a, b):
        x.axis("off")
plt.tight_layout(); plt.show()

## 7. Run on every image

Processes all images into one `uint8` array of shape (N, 224, 224) (843 MB for the full dataset) plus an index table. Takes a few minutes in Colab. Set `SAVE_DIR` to keep the result on Drive.

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/oct_preprocessed" if IN_COLAB else "preprocessed"

N = len(df)
images = np.zeros((N, SIZE, SIZE), dtype=np.uint8)
infos = []
t0 = time.time()
for i, p in enumerate(tqdm(df.path, desc="preprocess")):
    images[i], inf = preprocess(load_gray(p))
    infos.append(inf)
idx = pd.concat([df.drop(columns=["path"]), pd.DataFrame(infos)], axis=1)
idx["span"] = idx.retina_bottom - idx.retina_top
idx["clipped"] = (idx.retina_top < idx.crop_top) | (idx.retina_bottom > idx.crop_bottom)
print(f"{N:,} images in {time.time() - t0:.0f}s -> array {images.shape}, {images.nbytes / 1e6:.0f} MB")

Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
np.save(Path(SAVE_DIR, "images.npy"), images)
idx.assign(row=np.arange(N)).to_csv(Path(SAVE_DIR, "index.csv"), index=False)
print("saved to", Path(SAVE_DIR).resolve())

## 8. Did it work?

- **Fallback rate:** images where the retina was not found and a centre crop was used (full dataset: 0%).
- **Retina clipped:** the detected retina extends past the 352-row window (full dataset: 0.4% Normal, 0.4% Drusen, 1.7% CNV).
- **Why 352 rows:** the share of images whose whole retina fits, with an 8-pixel margin, for each window height.

In [ ]:
summary = idx.groupby("class_folder").agg(images=("image_path", "size"), fallback=("fallback", "mean"),
                                          retina_clipped=("clipped", "mean"), median_retina_height=("span", "median"),
                                          median_crop_top=("crop_top", "median")).reindex(CLASSES)
display(summary.style.format({"fallback": "{:.1%}", "retina_clipped": "{:.1%}", "median_retina_height": "{:.0f}",
                              "median_crop_top": "{:.0f}"}))

fit = pd.DataFrame({"window height (rows)": h, "retina fits (8 px margin)": f"{(idx.span <= h - 16).mean():.1%}"}
                   for h in (256, 288, 320, 352, 384))
display(fit)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
for c, col in zip(CLASSES, ["#2a78d6", "#eb6834", "#1baf7a"]):
    ax[0].hist(idx.span[idx.class_folder == c].dropna(), bins=40, histtype="step", lw=2, color=col, label=c, density=True)
    ax[1].hist(idx.crop_top[idx.class_folder == c], bins=40, histtype="step", lw=2, color=col, label=c, density=True)
ax[0].axvline(ROI_HEIGHT - 16, color="k", ls="--", lw=1); ax[0].set_xlabel("detected retina height (rows)")
ax[1].set_xlabel("crop window top (row)")
ax[0].legend(frameon=False)
for a in ax:
    a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# random sample of the final 224 x 224 inputs
sample = rng.choice(N, min(16, N), replace=False)
fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for a, i in zip(axes.flat, sample):
    a.imshow(images[i], cmap="gray", vmin=0, vmax=255)
    a.set_title(f"{idx.patient_id[i]} / {idx.bscan_label[i]}", fontsize=8)
    a.axis("off")
plt.tight_layout(); plt.show()

**Next stages** (in the repository): patient-level split (03), training only on clean-normal B-scans of the training patients, then the anomaly-detection models (05–12) read this 224 × 224 cache.